In [ ]:
#!pip uninstall -y transformers
#!pip install git+https://github.com/huggingface/transformers.git@fix/lerobot_openpi

In [1]:
from datetime import datetime
import os

In [2]:
DATASET_REPO="gimarchetti/ur5-experiment-dataset" #@param {type:"string"}
DATASET_ROOT="./dataset/teleoperation_dataset" #@param {type:"string"}
POLICY_REPO="gimarchetti/ur5-experiment-pi05" #@param {type:"string"}
OUTPUT_DIR="./ckpt/ur5-experiment-pi05" #@param {type:"string"}
JOB_NAME="ur5-experiment-pi05"+datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MAX_TRAIN_STEPS=10000 #@param {type:"integer"}
CHUNK_SIZE=20 #@param {type:"integer"}
ACTION_STEPS=10 #@param {type:"integer"}
#EVAL_STEPS=1000
#SAVE_STEPS=1000
BATCH_SIZE=32 #@param {type:"integer"}
#LEARNING_RATE=5e-5
#WEIGHT_DECAY=0.01
#WARMUP_STEPS=500
#LOGGING_STEPS=100


In [ ]:
'''
Load environment configuration and initialize environments
'''
# Evaluation Configuration
TEST_EPISODES = 3 #@param {"type":"integer"}
MAX_EPISODE_STEPS = 50_000 #@param {"type":"string"}
TASK="Put the red cube in the open drawer then close the drawer"

In [3]:
import random
import numpy as np
import os
import torch
import json
from PIL import Image
from src.env.env import RILAB_OMY_ENV
from torchvision import transforms

from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from lerobot.processor import PolicyAction, PolicyProcessorPipeline
from lerobot.processor.converters import (
    batch_to_transition,
    policy_action_to_transition,
    transition_to_batch,
    transition_to_policy_action,
)
from lerobot.utils.constants import POLICY_POSTPROCESSOR_DEFAULT_NAME, POLICY_PREPROCESSOR_DEFAULT_NAME

import glfw

/opt/miniconda3/envs/mujoco2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Model

In [4]:
'''
Meta data is for loading dataset statistics and feature information
'''
#repo_id_or_path = 'Jeongeun/tutorial_v2_pi05' # Use this for loading pretrained model from the hub
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

policy = PI05Policy.from_pretrained(POLICY_REPO)
_ = policy.to(device)

# Set the number of action steps to run in the environment for one invocation of the policy
policy.config.n_action_steps = 10

The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


Loading model from: gimarchetti/ur5-experiment-pi05
✓ Loaded state dict from model.safetensors
	Missing key(s) in state_dict: "model.paligemma_with_expert.paligemma.model.language_model.embed_tokens.weight". 


**Note**: If you want to change number of actions in chunk to be executed, please change:  

```policy.config.n_action_steps = YOUR_DESIRED_NUMBER```

In [5]:
# Check normalization stats dimension
preprocessor = PolicyProcessorPipeline.from_pretrained(
                pretrained_model_name_or_path=POLICY_REPO,
                config_filename= f"{POLICY_PREPROCESSOR_DEFAULT_NAME}.json",
                overrides={"device_processor": {"device": device}},
                to_transition=batch_to_transition,
                to_output=transition_to_batch,
            )

for step in preprocessor.steps:
    if hasattr(step, "stats"):
        if "observation.state" in step.stats:
            print(f"Stats dimension for observation.state: {step.stats['observation.state']['mean'].shape}")

postprocessor = PolicyProcessorPipeline.from_pretrained(
                pretrained_model_name_or_path=POLICY_REPO,
                config_filename= f"{POLICY_POSTPROCESSOR_DEFAULT_NAME}.json",
                overrides={"device_processor": {"device": device}},
                to_transition=policy_action_to_transition,
                to_output=transition_to_policy_action,
            )

Stats dimension for observation.state: (7,)


In [ ]:
batch = {
    'observation.state': np.zeros((1, 7), dtype=np.float32),
    'observation.image': np.zeros((1, 3, 256, 256), dtype=np.float32),
    'observation.wrist_image': np.zeros((1, 3, 256, 256), dtype=np.float32),
    'task': [TASK]
}
batch = preprocessor(batch)  # to initialize the processors
_ = policy.select_action(batch)  # to initialize the model

## Load Environment

In [19]:
config_file_path = './configs/train_ur5.json'
with open(config_file_path) as f:
    env_conf = json.load(f)
omy_env = RILAB_OMY_ENV(cfg=env_conf, seed=0, 
                        action_type='joint', 
                        obs_type='joint_pos',
                        vis_mode = 'teleop')


-----------------------------------------------------------------------------
name:[tabletop_env] dt:[0.002] HZ:[500]
 n_qpos:[38] n_qvel:[35] n_qacc:[35] n_ctrl:[8]
 integrator:[RK4]

n_body:[36]
 [0/36] [world] mass:[0.00]kg
 [1/36] [front_object_table] mass:[1.00]kg
 [2/36] [camera] mass:[0.00]kg
 [3/36] [camera2] mass:[0.00]kg
 [4/36] [camera3] mass:[0.00]kg
 [5/36] [base] mass:[4.00]kg
 [6/36] [shoulder_link] mass:[3.70]kg
 [7/36] [upper_arm_link] mass:[8.39]kg
 [8/36] [forearm_link] mass:[2.27]kg
 [9/36] [wrist_1_link] mass:[1.22]kg
 [10/36] [wrist_2_link] mass:[1.22]kg
 [11/36] [wrist_3_link] mass:[0.19]kg
 [12/36] [camera_center] mass:[0.00]kg
 [13/36] [attachment] mass:[0.00]kg
 [14/36] [base_mount] mass:[0.15]kg
 [15/36] [gripper_base] mass:[0.78]kg
 [16/36] [tcp_link] mass:[0.00]kg
 [17/36] [right_driver] mass:[0.01]kg
 [18/36] [right_coupler] mass:[0.01]kg
 [19/36] [right_spring_link] mass:[0.02]kg
 [20/36] [right_follower] mass:[0.01]kg
 [21/36] [right_pad] mass:[0.00]kg


In [20]:
def get_default_transform():
    """
    Returns a torchvision transform that:
     Converts to a FloatTensor and scales pixel values [0,255] -> [0.0,1.0]
    """
    return transforms.Compose([
        transforms.ToTensor(),  # PIL [0–255] -> FloatTensor [0.0–1.0], shape C×H×W
    ])
IMG_TRANSFORM = get_default_transform()

## Evaluate

In [ ]:
'''
Run one evaluation episode
'''
def run_one_episode():
    omy_env.reset()
    policy.reset()
    observation = omy_env.get_observation()
    omy_env.env.tick = 0
    success = False
    while omy_env.env.is_viewer_alive() and omy_env.env.tick < MAX_EPISODE_STEPS:
        omy_env.step_env()
        if omy_env.env.loop_every(HZ = 20):
            success = omy_env.check_success()
            if success: break
            if omy_env.env.is_key_pressed_once(glfw.KEY_Z):
                break  # for debugging: press 'z' to end the episode
            agent_image, wrist_image = omy_env.grab_image(return_side=False)
            # # resize to 256x256
            frame = {
                "observation.state": observation[:7].astype(np.float32),
                'task': [TASK] #[env_conf['language_instruction']]
            }
            agent_image = Image.fromarray(agent_image)
            wrist_image = Image.fromarray(wrist_image)
            agent_image = agent_image.resize((256, 256))
            wrist_image = wrist_image.resize((256, 256))
            agent_image = IMG_TRANSFORM(agent_image)
            wrist_image = IMG_TRANSFORM(wrist_image)
            frame["observation.image"] = agent_image
            frame["observation.wrist_image"] = wrist_image
            # numpy to torch
            frame = {k: torch.tensor(v, dtype=torch.float32).unsqueeze(0) if isinstance(v, np.ndarray) else v for k, v in frame.items()}
            # pre-process the frame
            frame = preprocessor(frame)
            # select action
            action = policy.select_action(frame)
            # post-process the action
            action = postprocessor(action)
            action = action.squeeze(0).cpu().numpy()
            observation = omy_env.step(action, gripper_mode='continuous')
            omy_env.render()
    return success

In [22]:
'''
Run evaluation over multiple episodes
'''
results = []
for episode in range(TEST_EPISODES):
    success = run_one_episode()
    results.append(success)
    print(f"Episode {episode+1}/{TEST_EPISODES} - Success: {success}")
omy_env.env.close_viewer()
# log average success rate
avg_success = sum(results) / len(results)
print(f"Average Success Rate over {TEST_EPISODES} episodes: {avg_success*100:.2f}%")


['top', 'open']
DONE INITIALIZATION
Episode 1/3 - Success: False
['top', 'open']
DONE INITIALIZATION
Episode 2/3 - Success: False
['top', 'open']
DONE INITIALIZATION
Episode 3/3 - Success: False
Average Success Rate over 3 episodes: 0.00%
